# U and V montly means

In [1]:
import xarray as xr
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cf
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.patches as mpatches
from tqdm import tqdm

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
from dask.distributed import Client
client = Client(n_workers=4, threads_per_worker=3, memory_limit=15e9)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:45223/status,
Dashboard: http://127.0.0.1:45223/status,Workers: 4
Total threads: 12,Total memory: 55.88 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37635,Workers: 0
Dashboard: http://127.0.0.1:45223/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:35221,Total threads: 3
Dashboard: http://127.0.0.1:41185/status,Memory: 13.97 GiB
Nanny: tcp://127.0.0.1:35871,


In [4]:
ssh = xr.open_dataset('/work/bk1450/b383184/Amazon/Mercator/data/variables_c/tracers/SSH_1993-01c.nc')
ssh = ssh.compute()

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'sossheig' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


In [5]:
uu = xr.open_dataset('/work/bk1450/b383184/Amazon/Mercator/data/variables_c/UVW/U_1993-01c.nc').deptht.isel(deptht=slice(0,7))
uu

<xarray.DataArray 'deptht' (deptht: 7)> Size: 28B
array([0.494025, 1.541375, 2.645669, 3.819495, 5.078224, 6.440614, 7.92956 ],
      dtype=float32)
Coordinates:
  * deptht   (deptht) float32 28B 0.494 1.541 2.646 3.819 5.078 6.441 7.93
Attributes:
    standard_name:  depth
    long_name:      Vertical T levels
    units:          m
    positive:       down
    axis:           Z
    _ChunkSizes:    50

In [6]:
dz = np.diff(uu['deptht'].values, prepend=0)
dz = xr.DataArray(dz, dims='deptht', coords={'deptht': uu['deptht']})
# dz = dz.rename({'deptht':'deptht'})
dz

<xarray.DataArray (deptht: 7)> Size: 56B
array([0.49402538, 1.04735002, 1.10429311, 1.17382622, 1.25872898,
       1.36239052, 1.48894596])
Coordinates:
  * deptht   (deptht) float32 28B 0.494 1.541 2.646 3.819 5.078 6.441 7.93

### U weighted mean 0-8m

In [7]:
U = xr.open_mfdataset('/work/bk1450/b383184/Amazon/Mercator/data/variables_c/UVW/U_*.nc',chunks={'time_counter':-1, 'deptht': 7, 'y': 250, 'x': 600},
                        ).isel(deptht=slice(0, 7))
U = U.rename({"vozocrtx": "U"})
U

In [8]:
U.U.weighted(dz).mean('deptht')
U_m = U.U.weighted(dz).mean('deptht').resample(time_counter='MS').mean()
U_m.drop_encoding().to_zarr('U_m_0_7m.zarr', mode='w')

In [9]:
# xr.open_zarr('U_m_0_7m.zarr').to_netcdf('U_m_0_7m.nc')

### V weighted mean 0-8m

In [10]:
V = xr.open_mfdataset('/work/bk1450/b383184/Amazon/Mercator/data/variables_c/UVW/V_*.nc',chunks={'time_counter':-1, 'deptht': 7, 'y': 250, 'x': 600},
                        ).isel(deptht=slice(0, 7))
V = V.rename({"vomecrty": "V"})
V

<xarray.Dataset> Size: 140GB
Dimensions:       (time_counter: 7973, deptht: 7, y: 499, x: 1260)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 64kB 1993-01-01T12:00:00 ... ...
    nav_lon       (y, x) float32 3MB dask.array<chunksize=(250, 600), meta=np.ndarray>
    nav_lat       (y, x) float32 3MB dask.array<chunksize=(250, 600), meta=np.ndarray>
  * x             (x) float64 10kB 2.306e+03 2.307e+03 ... 3.564e+03 3.565e+03
  * y             (y) float64 4kB 1.375e+03 1.376e+03 ... 1.872e+03 1.873e+03
  * deptht        (deptht) float32 28B 0.494 1.541 2.646 3.819 5.078 6.441 7.93
Data variables:
    V             (time_counter, deptht, y, x) float32 140GB dask.array<chunksize=(31, 7, 250, 600), meta=np.ndarray>
Attributes:
    CDI:          Climate Data Interface version 2.2.4 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Fri Nov 07 00:01:55 2025: cdo setmissval,nan V_1993-01.nc /...
    CDO:          Climate Data Operators version 2.2.2 (https://mpimet.mpg.de...

In [11]:
V_m = V.V.weighted(dz).mean('deptht').resample(time_counter='MS').mean()
# V_m.drop_encoding().to_zarr('V_m_0_7m.zarr', mode='w')